# Sparse Release Drawdown-Control Target Handoff v1

## Purpose

The previous `sparse_release_failed_fold_interval_diagnostic_v1` notebook found that the selected PPO checkpoint improved 2022 relative return but widened MaxDD to `38.68%`. The follow-up `sparse_release_drawdown_budget_guard_screen_v1` notebook showed that post-hoc execution caps need near-cash exposure and still miss the `+30pp / <=15% MaxDD` gate.

This notebook converts the failed path interval into state-objective-compatible training targets. It is a handoff artifact for training/checkpoint selection, not a promotion review.

## External Basis

- Risk-sensitive MDP / CVaR objective: https://arxiv.org/abs/1506.02188
- Deep portfolio RL should optimize risk-aware sequential allocation, not just terminal return: https://arxiv.org/abs/1706.10059
- Qlib remains the project framework boundary for repeatable research and validation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL reinforces explicit DRL environment/training/replay separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- PBO / CSCV warns against selection-only overfit: https://doi.org/10.21314/JCF.2016.322

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveDrawdownControlTargetConfig,
    build_drawdown_control_training_targets,
    summarize_state_objective_training_targets,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

EXPERIMENT = 'sparse_release_drawdown_control_target_handoff_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
FAILED_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_failed_fold_interval_diagnostic_v1'
GUARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sparse_release_drawdown_budget_guard_screen_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0

In [2]:
ppo_daily = pd.read_csv(FAILED_DIR / 'ppo_selected_daily_2022.csv', parse_dates=['datetime'])
ppo_intervals = pd.read_csv(FAILED_DIR / 'ppo_worst_drawdown_intervals.csv')
feature_delta = pd.read_csv(FAILED_DIR / 'feature_delta_by_interval.csv')
failed_summary = json.loads((FAILED_DIR / 'summary.json').read_text())
guard_summary = json.loads((GUARD_DIR / 'summary.json').read_text())
guard_metrics = pd.read_csv(GUARD_DIR / 'aggregate_guard_metrics.csv')

input_evidence = {
    'failed_decision': failed_summary['decision'],
    'ppo_selected_relative_return': failed_summary['ppo_selected_metrics']['relative_return'],
    'ppo_selected_max_dd': failed_summary['ppo_selected_metrics']['max_dd'],
    'worst_interval_start': ppo_intervals.iloc[0]['start'],
    'worst_interval_end': ppo_intervals.iloc[0]['end'],
    'worst_interval_max_dd': float(ppo_intervals.iloc[0]['max_dd']),
    'guard_decision': guard_summary['selection']['decision'],
    'guard_next_mechanism': guard_summary['selection']['next_mechanism'],
    'best_guard_config': guard_summary['selection']['best_config']['config_name'],
    'best_guard_max_drawdown_gap': guard_summary['selection']['best_config']['max_drawdown_gap'],
    'best_guard_max_relative_gap': guard_summary['selection']['best_config']['max_relative_gap'],
    'best_guard_mean_avg_pos': guard_summary['selection']['best_config']['mean_avg_pos'],
}
input_evidence

{'failed_decision': 'ppo_failed_wf_2022_drawdown_interval_no_longer_training',
 'ppo_selected_relative_return': 41.77033285370087,
 'ppo_selected_max_dd': 38.68269354530396,
 'worst_interval_start': '2022-03-30',
 'worst_interval_end': '2022-10-14',
 'worst_interval_max_dd': 38.6827,
 'guard_decision': 'drawdown_budget_guard_screen_no_promotion',
 'guard_next_mechanism': 'move_drawdown_control_into_checkpoint_objective_or_state_conditioned_training',
 'best_guard_config': 'cvar_budget_10_15_cash',
 'best_guard_max_drawdown_gap': 0.7869898207450561,
 'best_guard_max_relative_gap': 4.007458010291202,
 'best_guard_mean_avg_pos': 0.05116526449456248}

In [3]:
config = StateObjectiveDrawdownControlTargetConfig(
    target_relative_return=TARGET_RELATIVE_RETURN_PP,
    target_max_dd=TARGET_MAX_DD_PCT,
    drawdown_state_threshold=0.10,
    downtrend_position_cap=0.25,
    min_position_delta=0.05,
    drawdown_gap_weight=10.0,
    interval_weight=5.0,
)

targets = build_drawdown_control_training_targets(
    ppo_daily,
    intervals=ppo_intervals,
    candidate='ppo_selected_sparse_release',
    window='wf_2022',
    config=config,
)

targets['fold'] = 'wf_2022'
targets['target_source'] = 'ppo_failed_drawdown_interval'
targets['target_relative_return_pp'] = TARGET_RELATIVE_RETURN_PP
targets['target_max_dd_pct'] = TARGET_MAX_DD_PCT

targets_path = OUT_DIR / 'drawdown_control_training_targets.csv'
targets.to_csv(targets_path, index=False)

target_summary = summarize_state_objective_training_targets(targets)
targets.head()

,datetime,candidate,window,cluster,teacher_position,rebound_target_position,risk_target_position,risk_cap_delta,objective_weight,walkforward_relative_weight,downtrend_drawdown_weight,rebound_participation_weight,cluster_drawdown_weight,window_relative_return,window_max_dd,window_relative_gap,window_drawdown_gap,path_drawdown,path_drawdown_gap,source_interval_rank,source_interval_start,source_interval_end,source_interval_max_dd,inside_failed_drawdown_interval,drawdown_control_state,cluster_relative_gap,cluster_participation_gap,cluster_drawdown_gap,is_rebound_cluster,is_downtrend_state,rebound_state,risk_state,nav,account,mom_5,mom_21,drawdown_63,vol_of_vol_21,fold,target_source,target_relative_return_pp,target_max_dd_pct
0,2022-01-03,ppo_selected_sparse_release,wf_2022,,0.250000,0.250000,0.250000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.682694,0.0,23.682694,0.0,0.0,0,,,0.0,False,False,0.0,0.0,0.0,False,False,False,False,1.0,1000000.0,-0.009199,0.174637,-0.012193,0.105463,wf_2022,ppo_failed_drawdown_interval,30.0,15.0
1,2022-01-04,ppo_selected_sparse_release,wf_2022,,0.232734,0.232734,0.232734,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.682694,0.0,23.682694,0.0,0.0,0,,,0.0,False,False,0.0,0.0,0.0,False,False,False,False,1.0,1000000.0,0.010356,0.165411,-0.026305,0.098579,wf_2022,ppo_failed_drawdown_interval,30.0,15.0
2,2022-01-05,ppo_selected_sparse_release,wf_2022,,0.221095,0.221095,0.221095,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.682694,0.0,23.682694,0.0,0.0,0,,,0.0,False,False,0.0,0.0,0.0,False,False,False,False,1.0,1000000.0,-0.095789,0.071316,-0.123030,0.083039,wf_2022,ppo_failed_drawdown_interval,30.0,15.0
3,2022-01-06,ppo_selected_sparse_release,wf_2022,,0.228940,0.228940,0.228940,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.682694,0.0,23.682694,0.0,0.0,0,,,0.0,False,False,0.0,0.0,0.0,False,False,False,False,1.0,1000000.0,-0.034704,-0.054782,-0.102069,0.079753,wf_2022,ppo_failed_drawdown_interval,30.0,15.0
4,2022-01-07,ppo_selected_sparse_release,wf_2022,,0.204569,0.204569,0.204569,0.0,1.0,0.0,0.0,0.0,0.0,0.0,38.682694,0.0,23.682694,0.0,0.0,0,,,0.0,False,False,0.0,0.0,0.0,False,False,False,False,1.0,1000000.0,-0.118079,-0.125219,-0.181532,0.074890,wf_2022,ppo_failed_drawdown_interval,30.0,15.0


In [4]:
state_slice_summary = (
    targets.groupby(['drawdown_control_state', 'is_downtrend_state', 'inside_failed_drawdown_interval'], dropna=False)
    .agg(
        rows=('datetime', 'count'),
        start=('datetime', 'min'),
        end=('datetime', 'max'),
        mean_teacher_position=('teacher_position', 'mean'),
        mean_risk_target_position=('risk_target_position', 'mean'),
        mean_risk_cap_delta=('risk_cap_delta', 'mean'),
        max_path_drawdown=('path_drawdown', 'max'),
        mean_objective_weight=('objective_weight', 'mean'),
        max_objective_weight=('objective_weight', 'max'),
    )
    .reset_index()
)
state_slice_summary['start'] = state_slice_summary['start'].dt.strftime('%Y-%m-%d')
state_slice_summary['end'] = state_slice_summary['end'].dt.strftime('%Y-%m-%d')
state_slice_summary.to_csv(OUT_DIR / 'state_slice_summary.csv', index=False)

feature_focus = feature_delta.sort_values('abs_delta_z', ascending=False).head(8).copy()
feature_focus.to_csv(OUT_DIR / 'drawdown_interval_feature_focus.csv', index=False)

risk_rows = targets[targets['is_downtrend_state']].copy()
summary_metrics = {
    'target_rows': int(len(targets)),
    'drawdown_control_rows': int(targets['drawdown_control_state'].sum()),
    'risk_cap_rows': int(targets['is_downtrend_state'].sum()),
    'failed_interval_rows': int(targets['inside_failed_drawdown_interval'].sum()),
    'mean_teacher_position': float(targets['teacher_position'].mean()),
    'mean_risk_target_position': float(targets['risk_target_position'].mean()),
    'risk_rows_mean_teacher_position': float(risk_rows['teacher_position'].mean()) if len(risk_rows) else 0.0,
    'risk_rows_mean_risk_target_position': float(risk_rows['risk_target_position'].mean()) if len(risk_rows) else 0.0,
    'risk_rows_mean_cap_delta': float(risk_rows['risk_cap_delta'].mean()) if len(risk_rows) else 0.0,
    'max_objective_weight': float(targets['objective_weight'].max()),
    'max_path_drawdown_pct': float(targets['path_drawdown'].max() * 100.0),
    'window_drawdown_gap_pp': float(targets['window_drawdown_gap'].max()),
    'top_feature_focus': feature_focus['feature'].head(5).tolist(),
}

state_slice_summary

,drawdown_control_state,is_downtrend_state,inside_failed_drawdown_interval,rows,start,end,mean_teacher_position,mean_risk_target_position,mean_risk_cap_delta,max_path_drawdown,mean_objective_weight,max_objective_weight
0,False,False,False,60,2022-01-03,2022-03-29,0.250358,0.250358,0.000000,0.000000,1.000000,1.0
1,True,False,True,45,2022-03-30,2022-08-04,0.249541,0.249541,0.000000,0.265023,1.000000,1.0
2,True,True,False,52,2022-10-17,2022-12-29,0.398317,0.250000,0.148317,0.378119,25.000000,25.0
3,True,True,True,93,2022-04-21,2022-10-14,0.407399,0.250000,0.157399,0.386827,22.587991,25.0


In [5]:
external_sources = [
    'https://arxiv.org/abs/1506.02188',
    'https://arxiv.org/abs/1706.10059',
    'https://github.com/microsoft/qlib',
    'https://arxiv.org/abs/2009.11189',
    'https://github.com/AI4Finance-Foundation/FinRL',
    'https://arxiv.org/abs/2011.09607',
    'https://doi.org/10.21314/JCF.2016.322',
]

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective'],
    'profile': 'sparse_release_bc_ppo_screen_v1',
    'source_experiment': 'sparse_release_failed_fold_interval_diagnostic_v1',
    'source_guard_experiment': 'sparse_release_drawdown_budget_guard_screen_v1',
    'decision': 'drawdown_control_target_handoff_no_promotion',
    'promotion_allowed': False,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'next_mechanism': 'train_or_checkpoint_score_state_objective_drawdown_targets',
    'required_terms': target_summary['required_terms'],
    'external_sources': external_sources,
    'sidecars': [
        'drawdown_control_training_targets.csv',
        'state_slice_summary.csv',
        'drawdown_interval_feature_focus.csv',
        'selection.json',
        'summary.json',
    ],
}

conclusion = [
    f"Generated {summary_metrics['target_rows']} drawdown-control target rows from the failed wf_2022 PPO path.",
    f"Drawdown-control state covers {summary_metrics['drawdown_control_rows']} rows; {summary_metrics['risk_cap_rows']} high-exposure rows require an explicit risk cap.",
    f"On capped risk rows, teacher position averages {summary_metrics['risk_rows_mean_teacher_position']:.3f} and risk_target_position averages {summary_metrics['risk_rows_mean_risk_target_position']:.3f}; mean cap delta is {summary_metrics['risk_rows_mean_cap_delta']:.3f}.",
    f"Max path drawdown in the source path is {summary_metrics['max_path_drawdown_pct']:.2f}%, leaving a {summary_metrics['window_drawdown_gap_pp']:.2f}pp gap to the <=15% target.",
    f"Execution-only guard remains diagnostic: best guard `{input_evidence['best_guard_config']}` still has full_gate=False and mean_avg_pos={input_evidence['best_guard_mean_avg_pos']:.3f}.",
    "Decision: no runtime promotion. Use this target table for state-conditioned training, BC warm start, or checkpoint-objective scoring before another PPO handoff.",
]

summary = {
    'experiment': EXPERIMENT,
    'decision': selection['decision'],
    'promotion_allowed': False,
    'input_evidence': input_evidence,
    'config': config.__dict__,
    'target_summary': target_summary,
    'summary_metrics': summary_metrics,
    'state_slice_summary': state_slice_summary.to_dict(orient='records'),
    'selection': selection,
    'conclusion': conclusion,
}

(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, ensure_ascii=False) + chr(10))
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False) + chr(10))

summary

{'experiment': 'sparse_release_drawdown_control_target_handoff_v1',
 'decision': 'drawdown_control_target_handoff_no_promotion',
 'promotion_allowed': False,
 'input_evidence': {'failed_decision': 'ppo_failed_wf_2022_drawdown_interval_no_longer_training',
  'ppo_selected_relative_return': 41.77033285370087,
  'ppo_selected_max_dd': 38.68269354530396,
  'worst_interval_start': '2022-03-30',
  'worst_interval_end': '2022-10-14',
  'worst_interval_max_dd': 38.6827,
  'guard_decision': 'drawdown_budget_guard_screen_no_promotion',
  'guard_next_mechanism': 'move_drawdown_control_into_checkpoint_objective_or_state_conditioned_training',
  'best_guard_config': 'cvar_budget_10_15_cash',
  'best_guard_max_drawdown_gap': 0.7869898207450561,
  'best_guard_max_relative_gap': 4.007458010291202,
  'best_guard_mean_avg_pos': 0.05116526449456248},
 'config': {'target_relative_return': 30.0,
  'target_max_dd': 15.0,
  'drawdown_state_threshold': 0.1,
  'downtrend_position_cap': 0.25,
  'min_position_de

## Notebook Conclusion

The notebook output above is the durable single-run conclusion. This card is not a runtime promotion. It creates the next training/checkpoint-selection target table from the failed 2022 drawdown path and keeps the special-case source-window logic inside the notebook.